In [1]:
import pandas as pd

df = pd.read_csv('car_fuel_efficiency_2026.csv')
df = df[['engine_displacement', 'horsepower', 'vehicle_weight',
         'model_year', 'fuel_efficiency_mpg']]


In [2]:
# Q1 — which column has missing values
print(df.isnull().sum())


engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64


In [3]:
# Q2 — median horsepower
print("Median horsepower:", df['horsepower'].median())

# Fuel efficiency tail check
print(df['fuel_efficiency_mpg'].describe())

Median horsepower: 254.0
count    10000.000000
mean        29.997700
std          2.930245
min         19.800000
25%         28.000000
50%         30.000000
75%         31.900000
max         41.200000
Name: fuel_efficiency_mpg, dtype: float64


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv('car_fuel_efficiency_2026.csv')
df = df[['engine_displacement', 'horsepower', 'vehicle_weight',
         'model_year', 'fuel_efficiency_mpg']]

n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
df_val   = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
df_test  = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('car_fuel_efficiency_2026.csv')
df = df[['engine_displacement', 'horsepower', 'vehicle_weight',
         'model_year', 'fuel_efficiency_mpg']]

n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
df_val   = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
df_test  = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

len(df_train), len(df_val), len(df_test)

(6000, 2000, 2000)

In [6]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [7]:
# Fill with 0 (in all three splits)
df_train_0 = df_train.copy()
df_val_0   = df_val.copy()
df_val_0['horsepower'] = df_val_0['horsepower'].fillna(0)

df_train_0['horsepower'] = df_train_0['horsepower'].fillna(0)

X_train = df_train_0[['engine_displacement', 'horsepower',
                      'vehicle_weight', 'model_year']].values
y_train = df_train_0['fuel_efficiency_mpg'].values

w0, w = train_linear_regression(X_train, y_train)

X_val = df_val_0[['engine_displacement', 'horsepower',
                  'vehicle_weight', 'model_year']].values
y_val = df_val_0['fuel_efficiency_mpg'].values

y_pred = w0 + X_val.dot(w)
score_zero = round(rmse(y_val, y_pred), 3)
print("Fill with 0   → RMSE:", score_zero)

Fill with 0   → RMSE: 2.205


In [8]:
# Compute mean on TRAINING ONLY
mean_hp = df_train['horsepower'].mean()
print("Training mean horsepower:", mean_hp)

df_train_m = df_train.copy()
df_val_m   = df_val.copy()

df_train_m['horsepower'] = df_train_m['horsepower'].fillna(mean_hp)
df_val_m['horsepower']   = df_val_m['horsepower'].fillna(mean_hp)

X_train = df_train_m[['engine_displacement', 'horsepower',
                      'vehicle_weight', 'model_year']].values
y_train = df_train_m['fuel_efficiency_mpg'].values

w0, w = train_linear_regression(X_train, y_train)

X_val = df_val_m[['engine_displacement', 'horsepower',
                  'vehicle_weight', 'model_year']].values
y_val = df_val_m['fuel_efficiency_mpg'].values

y_pred = w0 + X_val.dot(w)
score_mean = round(rmse(y_val, y_pred), 3)
print("Fill with mean → RMSE:", score_mean)

Training mean horsepower: 254.46338337605272
Fill with mean → RMSE: 2.202


In [11]:
def train_linear_regression_reg(X, y, r):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

In [12]:
# Fill with 0 in all splits
for d in (df_train, df_val, df_test):
    d['horsepower'] = d['horsepower'].fillna(0)

X_train = df_train[['engine_displacement', 'horsepower',
                    'vehicle_weight', 'model_year']].values
y_train = df_train['fuel_efficiency_mpg'].values

X_val = df_val[['engine_displacement', 'horsepower',
                'vehicle_weight', 'model_year']].values
y_val = df_val['fuel_efficiency_mpg'].values

In [13]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = w0 + X_val.dot(w)
    score = round(rmse(y_val, y_pred), 4)
    print(f"r = {r:>6}  →  RMSE = {score}")

r =      0  →  RMSE = 2.2053
r =   0.01  →  RMSE = 2.2058
r =    0.1  →  RMSE = 2.2241
r =      1  →  RMSE = 2.3492
r =      5  →  RMSE = 2.4094
r =     10  →  RMSE = 2.4195
r =    100  →  RMSE = 2.4292


In [15]:
import pandas as pd
import numpy as np

df = pd.read_csv('car_fuel_efficiency_2026.csv')
df = df[['engine_displacement', 'horsepower', 'vehicle_weight',
         'model_year', 'fuel_efficiency_mpg']]

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

scores = []

for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]].copy()
    df_val   = df.iloc[idx[n_train:n_train + n_val]].copy()

    # Fill missing horsepower with 0
    df_train['horsepower'] = df_train['horsepower'].fillna(0)
    df_val['horsepower']   = df_val['horsepower'].fillna(0)

    X_train = df_train[['engine_displacement', 'horsepower',
                        'vehicle_weight', 'model_year']].values
    y_train = df_train['fuel_efficiency_mpg'].values

    X_val = df_val[['engine_displacement', 'horsepower',
                    'vehicle_weight', 'model_year']].values
    y_val = df_val['fuel_efficiency_mpg'].values

    w0, w = train_linear_regression(X_train, y_train)
    y_pred = w0 + X_val.dot(w)

    score = rmse(y_val, y_pred)
    scores.append(score)
    print(f"seed = {seed}  →  RMSE = {score:.4f}")

std = np.std(scores)
print("\nScores:", scores)
print("Std:", round(std, 3))

seed = 0  →  RMSE = 2.2392
seed = 1  →  RMSE = 2.2021
seed = 2  →  RMSE = 2.1634
seed = 3  →  RMSE = 2.2044
seed = 4  →  RMSE = 2.2019
seed = 5  →  RMSE = 2.2458
seed = 6  →  RMSE = 2.2697
seed = 7  →  RMSE = 2.1908
seed = 8  →  RMSE = 2.2166
seed = 9  →  RMSE = 2.2070

Scores: [np.float64(2.239204143046126), np.float64(2.2021128210838845), np.float64(2.1634197787530947), np.float64(2.2043588768539224), np.float64(2.2018800943311554), np.float64(2.2457851509085134), np.float64(2.2697212079524043), np.float64(2.190794599933443), np.float64(2.216611201686444), np.float64(2.2070365928178957)]
Std: 0.029


In [16]:
import pandas as pd
import numpy as np

df = pd.read_csv('car_fuel_efficiency_2026.csv')
df = df[['engine_displacement', 'horsepower', 'vehicle_weight',
         'model_year', 'fuel_efficiency_mpg']]

def train_linear_regression_reg(X, y, r):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

# --- Split with seed 9 ---
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].copy()
df_val   = df.iloc[idx[n_train:n_train + n_val]].copy()
df_test  = df.iloc[idx[n_train + n_val:]].copy()

# --- Combine train + validation ---
df_full_train = pd.concat([df_train, df_val], ignore_index=True)

# --- Fill missing horsepower with 0 ---
df_full_train['horsepower'] = df_full_train['horsepower'].fillna(0)
df_test['horsepower']       = df_test['horsepower'].fillna(0)

# --- Features / targets ---
X_full = df_full_train[['engine_displacement', 'horsepower',
                        'vehicle_weight', 'model_year']].values
y_full = df_full_train['fuel_efficiency_mpg'].values

X_test = df_test[['engine_displacement', 'horsepower',
                  'vehicle_weight', 'model_year']].values
y_test = df_test['fuel_efficiency_mpg'].values

# --- Train with r = 0.001 ---
w0, w = train_linear_regression_reg(X_full, y_full, r=0.001)

# --- Evaluate on test ---
y_pred = w0 + X_test.dot(w)
score = round(rmse(y_test, y_pred), 3)
print("Test RMSE:", score)

Test RMSE: 2.236
